# 01 — Preparación de los datos

Prepara las ventanas exportadas desde Databricks para entrenar y evaluar TranAD y TranAD+ (apartados 4.2.1 y 4.2.2 de la memoria).

1. Carga los tres conjuntos (entrenamiento, validación y prueba).
2. Localiza las ventanas con valores imposibles en algún contador (magnitud mayor que 10⁶) y las separa en un conjunto propio.
3. Ajusta los escaladores solo con el entrenamiento y escala todos los conjuntos.
4. Guarda las ventanas escaladas, los escaladores y los metadatos de cada ventana.

**Entrada:** `RUTA_DRIVE/train`, `RUTA_DRIVE/val` y `RUTA_DRIVE/test` (formato descrito en el README).
**Salida:** carpeta `RUTA_PREPARADOS`, que leen el resto de notebooks.

Se ejecuta en CPU en unos minutos. Con los datos sintéticos de `datos_sinteticos/` funciona igual, aunque las cifras no coinciden con las de la memoria.

## 1. Entorno

En Colab, monta Google Drive y clona el repositorio para tener acceso a `src/`. En local, busca la carpeta `src/` subiendo desde el directorio actual.

In [ ]:
# --- Entorno: localiza src/ y, en Colab, monta Google Drive ---
import os
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/Williams-MP/TFM_VIU_2026.git"
RUTA_REPO_COLAB = "/content/TFM_VIU_2026"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(RUTA_REPO_COLAB):
        # Mientras el repositorio sea privado hace falta un token de GitHub.
        # Si es público, basta con dejar el campo vacío.
        from getpass import getpass
        token = getpass("Token de GitHub (vacío si el repositorio es público): ")
        url = URL_REPOSITORIO.replace("https://", f"https://{token}@") if token else URL_REPOSITORIO
        subprocess.run(["git", "clone", "-q", url, RUTA_REPO_COLAB], check=True)
    ruta_src = f"{RUTA_REPO_COLAB}/src"
else:
    # Ejecución local: se sube por las carpetas hasta encontrar src/.
    directorio = os.getcwd()
    while not os.path.isdir(os.path.join(directorio, "src")):
        padre = os.path.dirname(directorio)
        if padre == directorio:
            raise FileNotFoundError("No se encuentra la carpeta src/ del repositorio.")
        directorio = padre
    ruta_src = os.path.join(directorio, "src")

if ruta_src not in sys.path:
    sys.path.append(ruta_src)

from config import RUTA_DRIVE, RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP, NOMBRES_CONTADORES
from datos import escalar

print("Datos de entrada:", RUTA_DRIVE)
print("Salida:          ", RUTA_PREPARADOS)

## 2. Carga de las ventanas

Cada fila del Parquet es una ventana. La columna `secuencia` contiene 12 reportes de 13 valores; aquí se convierte en un array de forma `(n_ventanas, 12, 13)` por conjunto. En este paso no se transforma ningún valor.

In [ ]:
# --- Carga de los tres conjuntos y comprobación de su forma ---
import numpy as np
import pandas as pd

N_REPORTES, N_VARIABLES = 12, 13


def cargar_ventanas(nombre_conjunto):
    # Lee todos los Parquet de la carpeta del conjunto y apila las secuencias.
    df = pd.read_parquet(f"{RUTA_DRIVE}/{nombre_conjunto}")
    X = np.stack([np.stack(fila).astype(np.float64) for fila in df["secuencia"]])
    # Las secuencias se guardan aparte como array; el resto de columnas son metadatos.
    metadatos = df.drop(columns="secuencia").reset_index(drop=True)
    return metadatos, X


meta_train, X_train = cargar_ventanas("train")
meta_val, X_val = cargar_ventanas("val")
meta_test, X_test = cargar_ventanas("test")

for nombre, X in [("train", X_train), ("val", X_val), ("test", X_test)]:
    # Todas las ventanas deben tener 12 reportes de 13 variables.
    assert X.shape[1:] == (N_REPORTES, N_VARIABLES), f"{nombre}: forma inesperada {X.shape}"
    print(f"{nombre}: {X.shape[0]:,} ventanas")

## 3. Ventanas con valores imposibles

Algunos reportes tienen en un contador valores de magnitud mayor que 10⁶ en un intervalo de unos 30 minutos, incluido un valor negativo causado por el desbordamiento de un entero de 32 bits. Son un error de medida, no un fallo de la red (apartado 4.2.1 de la memoria).

Se marcan las ventanas con al menos un valor así en cualquiera de los 11 contadores. El límite de 10⁶ es el mismo que define V02. Se trabaja con los valores sin escalar.

In [ ]:
# --- Localización de las ventanas con valores imposibles ---
UMBRAL_VALOR_IMPOSIBLE = 1_000_000


def marcar_valores_imposibles(X, nombre):
    # True en las ventanas con algún contador de magnitud mayor que el umbral.
    mascara = np.any(np.abs(X[:, :, IDX_CONTADORES]) > UMBRAL_VALOR_IMPOSIBLE, axis=(1, 2))
    print(f"{nombre}: {int(mascara.sum()):,} de {X.shape[0]:,} ventanas ({mascara.mean():.3%})")
    return mascara


print("--- Ventanas con valores imposibles, por conjunto ---")
mascara_train = marcar_valores_imposibles(X_train, "train")
mascara_val = marcar_valores_imposibles(X_val, "val")
mascara_test = marcar_valores_imposibles(X_test, "test")

print("\n--- Contadores afectados (entrenamiento) ---")
for idx, nombre in zip(IDX_CONTADORES, NOMBRES_CONTADORES):
    n_ventanas = int(np.any(np.abs(X_train[:, :, idx]) > UMBRAL_VALOR_IMPOSIBLE, axis=1).sum())
    if n_ventanas > 0:
        print(f"  {nombre}: {n_ventanas:,} ventanas, máximo absoluto = {np.abs(X_train[:, :, idx]).max():,.0f}")

print("\n--- Dispositivos afectados (entrenamiento) ---")
print(f"{meta_train.loc[mascara_train, 'device_pseudo_id'].nunique():,} de "
      f"{meta_train['device_pseudo_id'].nunique():,} dispositivos")

## 4. Exclusión y escalado

Las ventanas con valores imposibles salen de los tres conjuntos y se reúnen en uno propio, que sirve para comprobar si los detectores las marcan como anómalas (apartado 5.1.4).

El escalado sigue la ecuación (2) de la memoria:

- **Contadores:** logaritmo con signo, sign(x)·log(1+|x|), y después mínimo-máximo. El signo solo importa en el conjunto de valores imposibles, que contiene un valor negativo; para el resto equivale a log(1+x).
- **V01:** solo mínimo-máximo.
- **V02:** sin cambios, porque ya vale 0 o 1.

Los dos escaladores se ajustan solo con el entrenamiento ya filtrado, para que validación y prueba no influyan en él. El resultado queda en [0, 1] en entrenamiento, el mismo rango que la salida sigmoide de TranAD.

In [ ]:
# --- Exclusión de las ventanas con valores imposibles y escalado ---
from sklearn.preprocessing import MinMaxScaler

# 1. Conjuntos sin valores imposibles y conjunto aparte con ellos.
X_train_limpio, X_val_limpio, X_test_limpio = X_train[~mascara_train], X_val[~mascara_val], X_test[~mascara_test]
X_valores_imposibles = np.concatenate([X_train[mascara_train], X_val[mascara_val], X_test[mascara_test]])

# Metadatos alineados con cada conjunto, para identificar después cada ventana por su posición.
meta_train_limpio = meta_train[~mascara_train].reset_index(drop=True)
meta_val_limpio = meta_val[~mascara_val].reset_index(drop=True)
meta_test_limpio = meta_test[~mascara_test].reset_index(drop=True)
meta_valores_imposibles = pd.concat(
    [meta_train[mascara_train].assign(conjunto="train"),
     meta_val[mascara_val].assign(conjunto="val"),
     meta_test[mascara_test].assign(conjunto="test")],
    ignore_index=True,
)

print("--- Conjuntos tras la exclusión ---")
for nombre, X in [("train", X_train_limpio), ("val", X_val_limpio), ("test", X_test_limpio),
                  ("valores imposibles", X_valores_imposibles)]:
    print(f"{nombre}: {X.shape}")

# 2. Ajuste de los escaladores solo con el entrenamiento filtrado.
train_2d = X_train_limpio.reshape(-1, N_VARIABLES)
escalador_contadores = MinMaxScaler().fit(np.sign(train_2d[:, IDX_CONTADORES]) * np.log1p(np.abs(train_2d[:, IDX_CONTADORES])))
escalador_gap = MinMaxScaler().fit(train_2d[:, [IDX_GAP]])

# 3. Escalado de todos los conjuntos con los mismos escaladores (src/datos.py).
conjuntos_escalados = {
    nombre: escalar(X, escalador_contadores, escalador_gap, IDX_CONTADORES, IDX_GAP)
    for nombre, X in [("train", X_train_limpio), ("val", X_val_limpio), ("test", X_test_limpio),
                      ("valores_imposibles", X_valores_imposibles)]
}

print("\n--- Comprobación del escalado ---")
for nombre, X in conjuntos_escalados.items():
    if len(X):
        print(f"{nombre}: mín={X.min():.3f}, máx={X.max():.3f}, media={X.mean():.3f}")

## 5. Guardado

Se guardan en `RUTA_PREPARADOS`:

- `ventanas_escaladas.npz`: los cuatro conjuntos escalados (`train`, `val`, `test` y `valores_imposibles`).
- `escalador_contadores.pkl` y `escalador_gap.pkl`: los escaladores, para desescalar las ventanas o escalar datos nuevos de la misma forma.
- `metadatos_<conjunto>.parquet`: dispositivo, sesión, ventana, inicio y fin de cada ventana, en el mismo orden que en el `.npz`.

Así el resto de notebooks parte de aquí sin repetir la carga ni el escalado.

In [ ]:
# --- Guardado de los conjuntos preparados, los escaladores y los metadatos ---
import joblib

os.makedirs(RUTA_PREPARADOS, exist_ok=True)

np.savez_compressed(f"{RUTA_PREPARADOS}/ventanas_escaladas.npz", **conjuntos_escalados)
joblib.dump(escalador_contadores, f"{RUTA_PREPARADOS}/escalador_contadores.pkl")
joblib.dump(escalador_gap, f"{RUTA_PREPARADOS}/escalador_gap.pkl")

for nombre, meta in [("train", meta_train_limpio), ("val", meta_val_limpio), ("test", meta_test_limpio),
                     ("valores_imposibles", meta_valores_imposibles)]:
    meta.to_parquet(f"{RUTA_PREPARADOS}/metadatos_{nombre}.parquet", index=False)

print("Guardado en:", RUTA_PREPARADOS)
print(sorted(os.listdir(RUTA_PREPARADOS)))

## 6. Comprobación con las cifras de la memoria

Con los datos originales, el número de ventanas debe coincidir con el de la memoria (Tabla 5 y apartado 5.1.4). Con otros datos, esta celda solo informa de las diferencias.

In [ ]:
# --- Comparación con las cifras publicadas en la memoria ---
ESPERADO = {"train": 562_875, "val": 113_034, "test": 88_126, "valores_imposibles": 885}

for nombre, n_esperado in ESPERADO.items():
    n_obtenido = len(conjuntos_escalados[nombre])
    marca = "coincide" if n_obtenido == n_esperado else "distinto (normal si no son los datos originales)"
    print(f"{nombre}: {n_obtenido:,} ventanas, memoria: {n_esperado:,} -> {marca}")